# EASE — Embarrassingly Shallow Autoencoder

Implements EASE (Steck 2019) using numpy/scipy and runs it through the same sparsity experiments as SVD and content-based.

Note: `min_ratings=50` filter and float32 throughout to stay within 16 GB RAM. OOM levels recorded as NaN.

In [1]:
import gc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.linalg as linalg
import random
import warnings
import os

from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

DATA_DIR   = '../../ml-25m/'
OUTPUT_DIR = '../outputs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED  = 42
N_EVAL_USERS = 500
N_BOOTSTRAP  = 1_000
TOP_N        = 10
LAMBDA_EASE  = 500.0
MIN_RATINGS  = 50   # raised from 5 → aggressively shrinks Gram matrix on 16 GB

print('Imports OK.')
print(f'MIN_RATINGS={MIN_RATINGS}  LAMBDA_EASE={LAMBDA_EASE}')

Imports OK.
MIN_RATINGS=50  LAMBDA_EASE=500.0


In [2]:
class EASE:
    """
    Embarrassingly Shallow Autoencoder for Recommendation (Steck 2019).

    Closed-form item-item weight matrix via regularised Gram matrix inversion.
    Uses only numpy and scipy — no deep learning framework required.

    Memory design
    -------------
    - X (interaction matrix) is float32 throughout.
    - G (Gram matrix) is computed and stored as float32; scipy.linalg.inv
      dispatches to LAPACK SGETRF/SGETRI (single precision), halving the
      memory vs float64 at the cost of minor precision loss.
    - gc.collect() is called after each del to return memory to the OS
      immediately rather than waiting for Python's next GC cycle.
    - B is stored as float32.
    """

    def __init__(self, lambda_: float = 500.0, min_ratings: int = 50):
        self.lambda_ = lambda_
        self.min_ratings = min_ratings
        self.B = None            # float32 item-item weight matrix
        self.item_ids = None     # filtered item IDs aligned with B columns
        self.item_to_idx = {}    # item_id → column index in B

    def fit(self, X: sp.csr_matrix, item_ids: np.ndarray) -> 'EASE':
        """
        Fit the closed-form EASE solution.

        Parameters
        ----------
        X         : csr_matrix (n_users, n_items), float32, values = rating/5.0
        item_ids  : int64 array of length n_items, aligned with columns of X
        """
        # step 1: filter items with < min_ratings
        n_ratings_per_item = np.asarray((X > 0).sum(axis=0)).ravel()
        mask   = n_ratings_per_item >= self.min_ratings
        n_kept = int(mask.sum())

        if n_kept == 0:
            warnings.warn(
                f'EASE.fit: no items passed min_ratings={self.min_ratings}. '
                'B remains None; recommend() will return empty lists.'
            )
            return self

        X_f = X[:, mask]   # (n_users, n_kept), still float32 sparse
        self.item_ids    = item_ids[mask].copy()
        self.item_to_idx = {int(iid): i for i, iid in enumerate(self.item_ids)}

        # step 2: Gram matrix G = X^T X  (float32)
        G = np.asarray((X_f.T @ X_f).todense(), dtype=np.float32)
        del X_f
        gc.collect()

        # step 3: L2 regularisation
        G[np.diag_indices(n_kept)] += np.float32(self.lambda_)

        # step 4: Invert  (float32 → LAPACK SGETRF/SGETRI)
        # scipy.linalg.inv preserves float32 dtype via single-precision LAPACK.
        # peak memory: G (n_kept^2 * 4 B) + P (n_kept^2 * 4 B) = n_kept^2 * 8 B.
        P = linalg.inv(G)
        del G
        gc.collect()

        # step 5: Compute B
        # B_ij = -P_ij / P_jj  (divide each column j by its diagonal entry)
        # B_jj = 0             (no self-loops)
        diag_P = np.diag(P).copy()          # (n_kept,) float32
        B = -P / diag_P[np.newaxis, :]      # broadcast; peak = P + B = 2 * n_kept^2 * 4 B
        np.fill_diagonal(B, 0.0)
        del P
        gc.collect()

        self.B = B.astype(np.float32)       # already float32; cast is a no-op safety net
        del B
        gc.collect()
        return self

    def recommend(self,
                  user_train_ratings: dict,
                  seen_ids: set,
                  n: int = 10) -> list:
        """
        Return top-N item IDs for a user.

        Parameters
        ----------
        user_train_ratings : {item_id: raw_rating} from the training split
        seen_ids           : set of item IDs to exclude
        """
        if self.B is None or self.item_ids is None:
            return []

        r = np.zeros(len(self.item_ids), dtype=np.float32)
        for iid, rating in user_train_ratings.items():
            idx = self.item_to_idx.get(int(iid))
            if idx is not None:
                r[idx] = float(rating) / 5.0

        scores = r @ self.B   # one matrix-vector multiply

        ranked = sorted(
            ((int(self.item_ids[i]), float(scores[i]))
             for i in range(len(self.item_ids))
             if int(self.item_ids[i]) not in seen_ids),
            key=lambda x: x[1],
            reverse=True,
        )
        return [mid for mid, _ in ranked[:n]]


print('EASE class defined.')

EASE class defined.


In [3]:
def ndcg_at_k(recommended, relevant_ratings, k=10):
    if not relevant_ratings:
        return 0.0
    dcg = sum(
        (relevant_ratings[item] / 5.0) / np.log2(rank + 2)
        for rank, item in enumerate(recommended[:k])
        if item in relevant_ratings
    )
    ideal_rels = sorted(relevant_ratings.values(), reverse=True)[:k]
    idcg = sum(
        (rel / 5.0) / np.log2(rank + 2)
        for rank, rel in enumerate(ideal_rels)
    )
    return dcg / idcg if idcg > 0 else 0.0


def bootstrap_ci(values, n_boot=N_BOOTSTRAP, ci=0.95):
    if len(values) == 0:
        return (np.nan, np.nan)
    rng  = np.random.default_rng(RANDOM_SEED)
    arr  = np.asarray(values, dtype=float)
    boot = [np.mean(rng.choice(arr, size=len(arr), replace=True))
            for _ in range(n_boot)]
    alpha = (1 - ci) / 2
    return (float(np.percentile(boot, alpha * 100)),
            float(np.percentile(boot, (1 - alpha) * 100)))


def build_sparse_matrix(df):
    """
    Build a float32 CSR user-item matrix.
    Returns (X, item_ids_array, user_to_row_dict).
    """
    all_item_ids = sorted(df['movieId'].unique())
    all_user_ids = sorted(df['userId'].unique())
    item_to_col  = {int(iid): i for i, iid in enumerate(all_item_ids)}
    user_to_row  = {int(uid): i for i, uid in enumerate(all_user_ids)}

    rows = df['userId'].map(user_to_row).values.astype(np.int32)
    cols = df['movieId'].map(item_to_col).values.astype(np.int32)
    vals = (df['rating'].values / 5.0).astype(np.float32)   # float32 throughout

    X = sp.csr_matrix(
        (vals, (rows, cols)),
        shape=(len(all_user_ids), len(all_item_ids)),
        dtype=np.float32,
    )
    item_ids = np.array(all_item_ids, dtype=np.int64)
    return X, item_ids, user_to_row


def memory_estimate(X, min_ratings=MIN_RATINGS):
    """
    Print estimated item count and Gram matrix memory before fitting EASE.
    Uses float32 sizing (4 bytes/element) since fit() now uses float32.
    Peak = 2× Gram size (G present while computing P, then B while P exists).
    """
    n_per_item = np.asarray((X > 0).sum(axis=0)).ravel()
    n_kept     = int((n_per_item >= min_ratings).sum())
    gram_gb    = n_kept ** 2 * 4 / 1e9          # float32 Gram matrix
    peak_gb    = gram_gb * 2                     # G + P simultaneously
    print(f'  [MEM] Items passing min_ratings={min_ratings} filter : {n_kept:,}')
    print(f'  [MEM] Gram matrix (float32, {n_kept}×{n_kept})       : {gram_gb:.2f} GB')
    print(f'  [MEM] Peak estimate (G + P simultaneously)           : {peak_gb:.2f} GB')
    return n_kept


print('Helpers defined.')

Helpers defined.


In [4]:
print('Loading full 25M ratings...')
full_ratings = pd.read_csv(DATA_DIR + 'ratings.csv')
print(f'  {len(full_ratings):,} ratings loaded')

FULL_CATALOGUE_SIZE = 62_423

# fixed tail IDs for Experiment B (from full dataset — constant across levels)
print('Computing fixed tail IDs...')
_fp_full = (
    full_ratings.groupby('movieId')['rating']
    .count()
    .reset_index(name='rating_count')
    .sort_values('rating_count', ascending=False)
    .reset_index(drop=True)
)
_fp_full['cumpct'] = (
    _fp_full['rating_count'].cumsum() / _fp_full['rating_count'].sum() * 100
)
_cutoff_full   = (_fp_full['cumpct'] >= 80).idxmax()
tail_ids_fixed = set(_fp_full.loc[_cutoff_full + 1:, 'movieId'])
print(f'  Tail films : {len(tail_ids_fixed):,}')

Loading full 25M ratings...
  25,000,095 ratings loaded
Computing fixed tail IDs...
  Tail films : 56,718


---
## Experiment A — Growing Platform (Random Rating Subsample)

Uniformly random fraction of all 25M ratings at each level; catalogue shrinks
alongside data.  All six levels attempted; OOM levels caught and recorded as NaN.

In [5]:
SPARSITY_LEVELS_A = [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0]
N_RATINGS_A       = [2_500, 25_000, 250_000, 2_500_000, 12_500_000, 25_000_000]

results_A = []

for sparsity, n_target in tqdm(
    zip(SPARSITY_LEVELS_A, N_RATINGS_A), total=len(SPARSITY_LEVELS_A), desc='Exp A'
):
    n_actual = min(n_target, len(full_ratings))
    subset   = full_ratings.sample(n=n_actual, random_state=RANDOM_SEED)

    print(f'\n{"="*62}')
    print(f'EXP A  sparsity={sparsity}  n_ratings={len(subset):,}')
    print(f'{"="*62}')
    print(f'  Unique movies : {subset["movieId"].nunique():,}')
    print(f'  Unique users  : {subset["userId"].nunique():,}')

    train_df, test_df = train_test_split(
        subset, test_size=0.2, random_state=RANDOM_SEED
    )

    X_train, item_ids_arr, _ = build_sparse_matrix(train_df)
    print(f'  Train matrix  : {X_train.shape}  nnz={X_train.nnz:,}  dtype={X_train.dtype}')

    # memory estimate before attempting the inversion
    n_kept_est = memory_estimate(X_train, MIN_RATINGS)

    # fit EASE — wrapped in MemoryError guard
    ease = EASE(lambda_=LAMBDA_EASE, min_ratings=MIN_RATINGS)
    try:
        ease.fit(X_train, item_ids_arr)
    except MemoryError:
        msg = (
            f'MemoryError at sparsity={sparsity} '
            f'(est. {n_kept_est:,} items, Gram ~{n_kept_est**2*4/1e9:.1f} GB). '
            'Recording NaN.'
        )
        warnings.warn(msg)
        print(f'  !! {msg}')
        results_A.append(dict(
            sparsity=sparsity, n_ratings=len(subset),
            ndcg=np.nan, ci_lower=np.nan, ci_upper=np.nan,
        ))
        del X_train; gc.collect()
        continue
    finally:
        del X_train; gc.collect()

    if ease.B is None:
        print('  No items passed filter — recording NaN.')
        results_A.append(dict(
            sparsity=sparsity, n_ratings=len(subset),
            ndcg=np.nan, ci_lower=np.nan, ci_upper=np.nan,
        ))
        continue

    print(f'  EASE B matrix : {len(ease.item_ids):,} items  '
          f'({ease.B.nbytes / 1e6:.0f} MB at float32)')

    # 500 evaluation users
    eval_users = list(set(train_df['userId'].unique()) &
                      set(test_df['userId'].unique()))
    random.seed(RANDOM_SEED)
    eval_sample = random.sample(eval_users, min(N_EVAL_USERS, len(eval_users)))
    print(f'  Evaluation users : {len(eval_sample)}')

    test_lookup = (
        test_df.groupby('userId')
        .apply(lambda x: dict(zip(x['movieId'].astype(int), x['rating'])))
        .to_dict()
    )
    train_ratings_lookup = (
        train_df.groupby('userId')
        .apply(lambda x: dict(zip(x['movieId'].astype(int), x['rating'])))
        .to_dict()
    )
    train_seen_lookup = (
        train_df.groupby('userId')['movieId'].apply(set).to_dict()
    )

    ndcg_vals = []
    for uid in eval_sample:
        relevant = test_lookup.get(uid, {})
        if not relevant:
            continue
        recs = ease.recommend(
            train_ratings_lookup.get(uid, {}),
            train_seen_lookup.get(uid, set()),
            n=TOP_N,
        )
        ndcg_vals.append(ndcg_at_k(recs, relevant))

    mean_ndcg    = float(np.mean(ndcg_vals)) if ndcg_vals else np.nan
    ci_lo, ci_hi = bootstrap_ci(ndcg_vals)
    print(f'  NDCG@10 = {mean_ndcg:.4f}  CI=[{ci_lo:.4f}, {ci_hi:.4f}]  '
          f'(n={len(ndcg_vals)} users)')

    results_A.append(dict(
        sparsity=sparsity,
        n_ratings=len(subset),
        ndcg=round(mean_ndcg, 4),
        ci_lower=round(ci_lo, 4),
        ci_upper=round(ci_hi, 4),
    ))
    del ease; gc.collect()

print('\n✓ Experiment A complete.')

Exp A:   0%|          | 0/6 [00:00<?, ?it/s]


EXP A  sparsity=0.0001  n_ratings=2,500
  Unique movies : 1,559
  Unique users  : 2,436
  Train matrix  : (1959, 1306)  nnz=2,000  dtype=float32
  [MEM] Items passing min_ratings=50 filter : 0
  [MEM] Gram matrix (float32, 0×0)       : 0.00 GB
  [MEM] Peak estimate (G + P simultaneously)           : 0.00 GB
  No items passed filter — recording NaN.


/var/folders/c1/3_h3p94d34j2jnnt4_zzlhgw0000gn/T/ipykernel_67808/135668108.py:41: UserWarning: EASE.fit: no items passed min_ratings=50. B remains None; recommend() will return empty lists.
  warnings.warn(



EXP A  sparsity=0.001  n_ratings=25,000
  Unique movies : 5,701
  Unique users  : 20,056
  Train matrix  : (16643, 5145)  nnz=20,000  dtype=float32
  [MEM] Items passing min_ratings=50 filter : 10
  [MEM] Gram matrix (float32, 10×10)       : 0.00 GB
  [MEM] Peak estimate (G + P simultaneously)           : 0.00 GB
  EASE B matrix : 10 items  (0 MB at float32)
  Evaluation users : 500


/var/folders/c1/3_h3p94d34j2jnnt4_zzlhgw0000gn/T/ipykernel_67808/3861819341.py:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: dict(zip(x['movieId'].astype(int), x['rating'])))
/var/folders/c1/3_h3p94d34j2jnnt4_zzlhgw0000gn/T/ipykernel_67808/3861819341.py:74: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: dict(zip(x['movieId'].astype(int), x['rating'])))


...

  NDCG@10 = 0.0346  CI=[0.0261, 0.0433]  (n=500 users)

EXP A  sparsity=0.5  n_ratings=12,500,000
  Unique movies : 51,124
  Unique users  : 162,541
  Train matrix  : (162541, 48360)  nnz=10,000,000  dtype=float32
  [MEM] Items passing min_ratings=50 filter : 9,571
  [MEM] Gram matrix (float32, 9571×9571)       : 0.37 GB
  [MEM] Peak estimate (G + P simultaneously)           : 0.73 GB
  EASE B matrix : 9,571 items  (366 MB at float32)
  Evaluation users : 500


/var/folders/c1/3_h3p94d34j2jnnt4_zzlhgw0000gn/T/ipykernel_67808/3861819341.py:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: dict(zip(x['movieId'].astype(int), x['rating'])))
/var/folders/c1/3_h3p94d34j2jnnt4_zzlhgw0000gn/T/ipykernel_67808/3861819341.py:74: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: dict(zip(x['movieId'].astype(int), x['rating'])))


  NDCG@10 = 0.1328  CI=[0.1194, 0.1462]  (n=500 users)

EXP A  sparsity=1.0  n_ratings=25,000,000
  Unique movies : 59,047
  Unique users  : 162,541
  Train matrix  : (162541, 56697)  nnz=20,000,000  dtype=float32
  [MEM] Items passing min_ratings=50 filter : 12,181
  [MEM] Gram matrix (float32, 12181×12181)       : 0.59 GB
  [MEM] Peak estimate (G + P simultaneously)           : 1.19 GB
  EASE B matrix : 12,181 items  (594 MB at float32)
  Evaluation users : 500


/var/folders/c1/3_h3p94d34j2jnnt4_zzlhgw0000gn/T/ipykernel_67808/3861819341.py:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: dict(zip(x['movieId'].astype(int), x['rating'])))
/var/folders/c1/3_h3p94d34j2jnnt4_zzlhgw0000gn/T/ipykernel_67808/3861819341.py:74: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: dict(zip(x['movieId'].astype(int), x['rating'])))


  NDCG@10 = 0.4346  CI=[0.4111, 0.4572]  (n=500 users)

✓ Experiment A complete.


In [6]:
ease_A_df = pd.DataFrame(results_A)
out_A = OUTPUT_DIR + 'ease_expA_results.csv'
ease_A_df.to_csv(out_A, index=False)
print(f'Saved → {out_A}')
print(ease_A_df.to_string(index=False))

Saved → ../outputs/ease_expA_results.csv
 sparsity  n_ratings   ndcg  ci_lower  ci_upper
   0.0001       2500    NaN       NaN       NaN
   0.0010      25000 0.0079    0.0025    0.0144
   0.0100     250000 0.0104    0.0051    0.0169
   0.1000    2500000 0.0346    0.0261    0.0433
   0.5000   12500000 0.1328    0.1194    0.1462
   1.0000   25000000 0.4346    0.4111    0.4572


---
## Experiment B — Fixed Catalogue (Per-Movie Rating Subsampling)

All 62,423 films present at every level. Sparsity applied per-movie (`max(1, floor(n_i * s))`). OOM levels caught and recorded as NaN.

In [ ]:
def fixed_catalogue_sample(df, sparsity, random_seed=RANDOM_SEED):
    """Per-movie subsample keeping ≥1 rating per film."""
    return (
        df.groupby('movieId', group_keys=False)
        .apply(lambda x: x.sample(
            n=max(1, int(len(x) * sparsity)),
            random_state=random_seed,
        ))
        .reset_index(drop=True)
    )


SPARSITY_LEVELS_B = [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0]

results_B = []

for sparsity in tqdm(SPARSITY_LEVELS_B, desc='Exp B'):

    subset_B = fixed_catalogue_sample(full_ratings, sparsity)
    avg_rpm  = len(subset_B) / subset_B['movieId'].nunique()

    print(f'\n{"="*62}')
    print(f'EXP B  sparsity={sparsity}  n_ratings={len(subset_B):,}')
    print(f'{"="*62}')
    print(f'  Unique movies       : {subset_B["movieId"].nunique():,}')
    print(f'  Avg ratings / movie : {avg_rpm:.2f}')

    train_df_B, test_df_B = train_test_split(
        subset_B, test_size=0.2, random_state=RANDOM_SEED
    )

    X_B, item_ids_B, _ = build_sparse_matrix(train_df_B)
    print(f'  Train matrix : {X_B.shape}  nnz={X_B.nnz:,}  dtype={X_B.dtype}')

    # memory estimate
    n_kept_est_B = memory_estimate(X_B, MIN_RATINGS)

    # fit EASE — MemoryError guard
    ease_B = EASE(lambda_=LAMBDA_EASE, min_ratings=MIN_RATINGS)
    try:
        ease_B.fit(X_B, item_ids_B)
    except MemoryError:
        msg = (
            f'MemoryError at sparsity={sparsity} '
            f'(est. {n_kept_est_B:,} items, Gram ~{n_kept_est_B**2*4/1e9:.1f} GB). '
            'Recording NaN.'
        )
        warnings.warn(msg)
        print(f'  !! {msg}')
        results_B.append(dict(
            sparsity=sparsity, n_ratings=len(subset_B),
            ndcg=np.nan, ci_lower=np.nan, ci_upper=np.nan,
        ))
        del X_B; gc.collect()
        continue
    finally:
        del X_B; gc.collect()

    if ease_B.B is None:
        print('  No items passed filter — recording NaN.')
        results_B.append(dict(
            sparsity=sparsity, n_ratings=len(subset_B),
            ndcg=np.nan, ci_lower=np.nan, ci_upper=np.nan,
        ))
        continue

    print(f'  EASE B matrix : {len(ease_B.item_ids):,} items  '
          f'({ease_B.B.nbytes / 1e6:.0f} MB at float32)')

    # 500 evaluation users
    eval_users_B = list(set(train_df_B['userId'].unique()) &
                        set(test_df_B['userId'].unique()))
    random.seed(RANDOM_SEED)
    eval_sample_B = random.sample(eval_users_B, min(N_EVAL_USERS, len(eval_users_B)))
    print(f'  Evaluation users : {len(eval_sample_B)}')

    test_lookup_B = (
        test_df_B.groupby('userId', group_keys=False)
        .apply(lambda x: dict(zip(x['movieId'].astype(int), x['rating'])))
        .to_dict()
    )
    train_ratings_lookup_B = (
        train_df_B.groupby('userId', group_keys=False)
        .apply(lambda x: dict(zip(x['movieId'].astype(int), x['rating'])))
        .to_dict()
    )
    train_seen_lookup_B = (
        train_df_B.groupby('userId')['movieId'].apply(set).to_dict()
    )

    ndcg_vals_B = []
    for uid in eval_sample_B:
        relevant = test_lookup_B.get(uid, {})
        if not relevant:
            continue
        recs = ease_B.recommend(
            train_ratings_lookup_B.get(uid, {}),
            train_seen_lookup_B.get(uid, set()),
            n=TOP_N,
        )
        ndcg_vals_B.append(ndcg_at_k(recs, relevant))

    mean_ndcg_B      = float(np.mean(ndcg_vals_B)) if ndcg_vals_B else np.nan
    ci_lo_B, ci_hi_B = bootstrap_ci(ndcg_vals_B)
    print(f'  NDCG@10 = {mean_ndcg_B:.4f}  CI=[{ci_lo_B:.4f}, {ci_hi_B:.4f}]  '
          f'(n={len(ndcg_vals_B)} users)')

    results_B.append(dict(
        sparsity=sparsity,
        n_ratings=len(subset_B),
        ndcg=round(mean_ndcg_B, 4),
        ci_lower=round(ci_lo_B, 4),
        ci_upper=round(ci_hi_B, 4),
    ))
    del ease_B; gc.collect()

print('\n✓ Experiment B complete.')

In [8]:
ease_B_df = pd.DataFrame(results_B)
out_B = OUTPUT_DIR + 'ease_expB_results.csv'
ease_B_df.to_csv(out_B, index=False)
print(f'Saved → {out_B}')
print(ease_B_df.to_string(index=False))

Saved → ../outputs/ease_expB_results.csv
 sparsity  n_ratings   ndcg  ci_lower  ci_upper
   0.0001      59418    NaN       NaN       NaN
   0.0010      75667 0.0016    0.0000    0.0040
   0.0100     288378 0.0103    0.0041    0.0180
   0.1000    2513626 0.0298    0.0219    0.0386
   0.5000   12494180    NaN       NaN       NaN
   1.0000   25000095    NaN       NaN       NaN


---
## Combined Comparison Table

EASE alongside SVD, content-based, and hybrid at every sparsity level for both
experiments. NaN = memory constraint; loaded from saved CSVs.

In [9]:
existing_A = pd.read_csv(OUTPUT_DIR + 'sparsity_expA_all_models.csv')
existing_B = pd.read_csv(OUTPUT_DIR + 'sparsity_expB_all_models.csv')
ease_A     = pd.read_csv(OUTPUT_DIR + 'ease_expA_results.csv').assign(model='ease')
ease_B     = pd.read_csv(OUTPUT_DIR + 'ease_expB_results.csv').assign(model='ease')

COMPARE_MODELS = ['svd', 'content_based', 'hybrid', 'ease']


def comparison_table(existing, ease_results, exp_label):
    if 'experiment' in existing.columns:
        existing = existing.drop(columns='experiment')

    base = existing[existing['model'].isin(COMPARE_MODELS)][
        ['sparsity', 'n_ratings', 'model', 'ndcg']
    ].copy()
    ease_rows = ease_results[['sparsity', 'n_ratings', 'model', 'ndcg']].copy()

    combined = pd.concat([base, ease_rows], ignore_index=True)
    pivot = (
        combined
        .pivot_table(index=['sparsity', 'n_ratings'], columns='model', values='ndcg')
        .reset_index()
        .sort_values('sparsity', ascending=False)
        .reset_index(drop=True)
    )

    cols = ['sparsity', 'n_ratings'] + [
        m for m in COMPARE_MODELS if m in pivot.columns
    ]
    pivot = pivot[cols]

    model_cols   = [m for m in COMPARE_MODELS if m in pivot.columns]
    pivot['winner'] = pivot[model_cols].idxmax(axis=1)

    print(f'\n=== Experiment {exp_label}: NDCG@10 ===')
    print(pivot.to_string(index=False))
    return pivot


table_A = comparison_table(existing_A, ease_A, 'A  (shrinking catalogue)')
table_B = comparison_table(existing_B, ease_B, 'B  (fixed 62,423-film catalogue)')


=== Experiment A  (shrinking catalogue): NDCG@10 ===
 sparsity  n_ratings    svd  content_based  hybrid   ease winner
   1.0000   25000000    NaN            NaN     NaN 0.4346   ease
   1.0000   25000095 0.0669         0.1877  0.2115    NaN hybrid
   0.5000   12500000    NaN            NaN     NaN 0.1328   ease
   0.5000   12500047 0.0218         0.0687  0.0772    NaN hybrid
   0.1000    2500000    NaN            NaN     NaN 0.0346   ease
   0.1000    2500009 0.0069         0.0172  0.0270    NaN hybrid
   0.0100     250000 0.0079         0.0010  0.0038 0.0104   ease
   0.0010      25000 0.0057         0.0020  0.0034 0.0079   ease
   0.0001       2500 0.0000         0.0000  0.0000    NaN    svd

=== Experiment B  (fixed 62,423-film catalogue): NDCG@10 ===
 sparsity  n_ratings    svd  content_based  hybrid   ease        winner
   1.0000   25000095 0.0586         0.1729  0.1922    NaN        hybrid
   0.5000   12494180 0.0190         0.0763  0.0774    NaN        hybrid
   0.1000    25136